# BERT and Sequence labelling

Tutorial of Computational Linguistics, National Chengchi University

*Chang-Yu Tsai, 2025.04.25*

- In this week, we will try:
  - to build a model based on the the pre-tranined BERT
  - to conduct sequence labelling with pre-trained BERT
  


## Set-up

- importing required packages

```
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import BertTokenizerFast, BertForTokenClassification
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

import json
```



In [69]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import BertTokenizerFast, BertForTokenClassification
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

import json

## Preprocessing

- download the dataset from `Github`

The dataset is collected from [Chinese HealthNER Corpus](https://github.com/NYCU-NLP/Chinese-HealthNER-Corpus). Chinese Healthcare Named Entity Recognition (HealthNER) Corpus is collected and annotated by [NYCU NLP Lab](https://ainlp.tw/).

```
!wget https://raw.githubusercontent.com/NYCU-NLP/Chinese-HealthNER-Corpus/a5eaca54376267cee7a015eb870f7f302517d813/test.json
```

In [70]:
!wget https://raw.githubusercontent.com/NYCU-NLP/Chinese-HealthNER-Corpus/a5eaca54376267cee7a015eb870f7f302517d813/test.json

--2025-04-25 03:27:01--  https://raw.githubusercontent.com/NYCU-NLP/Chinese-HealthNER-Corpus/a5eaca54376267cee7a015eb870f7f302517d813/test.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3235436 (3.1M) [application/octet-stream]
Saving to: ‘test.json.2’

test.json.2         100%[===================>]   3.08M  --.-KB/s    in 0.06s   

2025-04-25 03:27:01 (49.5 MB/s) - ‘test.json.2’ saved [3235436/3235436]



- reading the file

```
data = []
with open("test.json", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))
print("Row number:",len(data))
print("The first row:\n",data[0])

```

In [37]:
data = []
with open("test.json", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))
print("Row number:",len(data))
print("The first row:\n",data[0])

Row number: 2531
The first row:
 {'id': '00000', 'genre': 'ft', 'sentence': '雞蛋含有多種維生素，包括Ｄ和Ｋ，是骨骼健康生長不可缺少的成分，又有豐富蛋白質。', 'word': ['雞蛋', '含有', '多種', '維生素', '，', '包括', 'Ｄ', '和', 'Ｋ', '，', '是', '骨骼', '健康', '生長', '不可', '缺少', '的', '成分', '，', '又', '有', '豐富', '蛋白質', '。'], 'word_label': ['O', 'O', 'O', 'SUPP', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'BODY', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'CHEM', 'O'], 'character': ['雞', '蛋', '含', '有', '多', '種', '維', '生', '素', '，', '包', '括', 'Ｄ', '和', 'Ｋ', '，', '是', '骨', '骼', '健', '康', '生', '長', '不', '可', '缺', '少', '的', '成', '分', '，', '又', '有', '豐', '富', '蛋', '白', '質', '。'], 'character_label': ['O', 'O', 'O', 'O', 'O', 'O', 'B-SUPP', 'I-SUPP', 'I-SUPP', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-BODY', 'I-BODY', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-CHEM', 'I-CHEM', 'I-CHEM', 'O']}


- inspecting the key of the dataset

```
data[0].keys()
```

In [71]:
data[0].keys()

dict_keys(['id', 'genre', 'sentence', 'word', 'word_label', 'character', 'character_label'])

- taking a look at the dataset

```
data[0]['sentence']
# data[0]['character']
# data[0]['character_label']
```

In [73]:
# data[0]['sentence']
# data[0]['character']
data[0]['character_label']

['O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-SUPP',
 'I-SUPP',
 'I-SUPP',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-BODY',
 'I-BODY',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-CHEM',
 'I-CHEM',
 'I-CHEM',
 'O']

- extracting the data we are going to work on NER

```
sentences=[]
characters=[]
character_labels=[]
for i in range(len(data)):
  sentences.append(data[i]['sentence'])
  characters.append(data[i]['character'])
  character_labels.append(data[i]['character_label'])
```


In [74]:
sentences=[]
characters=[]
character_labels=[]
for i in range(len(data)):
  sentences.append(data[i]['sentence'])
  characters.append(data[i]['character'])
  character_labels.append(data[i]['character_label'])

- calculating the max length of the sentence for the padding later

```
length_list=[]
for char_list in characters:
  length=len(char_list)
  length_list.append(length)
max_len=max(length_list)
print('The maximum sentence length:', max_len)
```

In [75]:
length_list=[]
for char_list in characters:
  length=len(char_list)
  length_list.append(length)
max_len=max(length_list)
print('The maximum sentence length:', max_len)

The maximum sentence length: 249


### obtaining the IDs of labels
We will later encode labels as tensors, and at first, we need to creat the mapping list of IDs and labels.


- defining a function to create mapping lists

```
def label_and_id(label_seqs):
    all_labels = set()
    for seq in label_seqs:
        for label in seq:
            all_labels.add(label)

    # sorting
    label_list = sorted(all_labels)

    # building the mapping lists
    label2id = {}
    for i, label in enumerate(label_list):
        label2id[label] = i
    id2label = {}
    for label, i in label2id.items():
        id2label[i] = label

    return label2id, id2label
```

In [76]:
def label_and_id(label_seqs):
    all_labels = set()
    for seq in label_seqs:
        for label in seq:
            all_labels.add(label)

    # sorting
    label_list = sorted(all_labels)

    # building the mapping lists
    label2id = {}
    for i, label in enumerate(label_list):
        label2id[label] = i
    id2label = {}
    for label, i in label2id.items():
        id2label[i] = label

    return label2id, id2label

- creating the mapping lists

```
label2id, id2label=label_and_id(character_labels)
print(label2id)
print(id2label)
```

In [77]:
label2id, id2label=label_and_id(character_labels)
print(label2id)
print(id2label)

{'B-BODY': 0, 'B-CHEM': 1, 'B-DISE': 2, 'B-DRUG': 3, 'B-EXAM': 4, 'B-INST': 5, 'B-SUPP': 6, 'B-SYMP': 7, 'B-TIME': 8, 'B-TREAT': 9, 'I-BODY': 10, 'I-CHEM': 11, 'I-DISE': 12, 'I-DRUG': 13, 'I-EXAM': 14, 'I-INST': 15, 'I-SUPP': 16, 'I-SYMP': 17, 'I-TIME': 18, 'I-TREAT': 19, 'O': 20}
{0: 'B-BODY', 1: 'B-CHEM', 2: 'B-DISE', 3: 'B-DRUG', 4: 'B-EXAM', 5: 'B-INST', 6: 'B-SUPP', 7: 'B-SYMP', 8: 'B-TIME', 9: 'B-TREAT', 10: 'I-BODY', 11: 'I-CHEM', 12: 'I-DISE', 13: 'I-DRUG', 14: 'I-EXAM', 15: 'I-INST', 16: 'I-SUPP', 17: 'I-SYMP', 18: 'I-TIME', 19: 'I-TREAT', 20: 'O'}


### data splitting
Before encoding the texts, we split the data into the trainging set (70\%), the dev set(10\%), and the test set (20\%).

```
# Step 1: 70% for the training set and 30% for the remaining data
train_chars, temp_chars, train_labels, temp_labels = train_test_split(
    characters, character_labels, test_size=0.2, random_state=42
)

# Step 2: in the remaining data, 10% for the dev set and 20% for the test set
dev_chars, test_chars, dev_labels, test_labels = train_test_split(
    temp_chars, temp_labels, test_size=2/3, random_state=42
)
```

In [78]:
# Step 1: 70% for the training set and 30% for the remaining data
train_chars, temp_chars, train_labels, temp_labels = train_test_split(
    characters, character_labels, test_size=0.2, random_state=42
)

# Step 2: in the remaining data, 10% for the dev set and 20% for the test set
dev_chars, test_chars, dev_labels, test_labels = train_test_split(
    temp_chars, temp_labels, test_size=2/3, random_state=42
)

### text encoding
It is important to conduct **tokenising** and **aligning** for text encoding before we work on BERT.

#### tokenising

`tokenizer` in HuggingFace Transformers is a built-in utility that handles not only tokenisation, but also:
- Automatic padding: With `padding='max_length'`, it pads all sequences to the maximum length of the dataset.

- Tensor output: With `return_tensors="pt"`, it directly returns PyTorch tensors (no need to convert manually).

- initialising the `tokenizer`

```
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")
```

In [79]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")

- a quick try

```
quick_try=tokenizer(
      [list('大膽甄嬛！'),list('鬢邊的秋海棠不俗')],            # List of List of characters
      is_split_into_words=True,
      padding='max_length',
      max_length=max_len,
      truncation=True,
      return_tensors="pt"
  )
print(quick_try['input_ids'][0])
print(quick_try['attention_mask'][0])
```

In [80]:
list('大膽甄嬛！')

['大', '膽', '甄', '嬛', '！']

In [46]:
quick_try=tokenizer(
      [list('大膽甄嬛！'),list('鬢邊的秋海棠不俗')],            # List of List of characters
      is_split_into_words=True,
      padding='max_length',
      max_length=max_len,
      truncation=True,
      return_tensors="pt"
  )
print(quick_try['input_ids'][0])
print(quick_try['attention_mask'][0])

tensor([ 101, 1920, 5615, 4488, 2083, 8013,  102,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,   

- defining the function to encode the texts

  **👓 Note that we also output `alignment_ids_list`, which is used to perform the following alignment between tokens and labels.**
  
  - We rely on the `.word_ids()` method to determine the alignment between subword tokens and their corresponding original characters.
    - `word_ids` assigns an integer index for each subword, indicating which original token (i.e., character) it comes from.
    - Special tokens such as `[CLS]`, `[SEP]`, and `[PAD]` will be assigned `None`, since they do not correspond to any original token. These should be ignored during label alignment.


```
def character_encode(characters, character_labels, label2id):
  encodings = tokenizer(
      characters,                    # List of List of characters
      is_split_into_words=True,
      padding='max_length',
      max_length=max_len,
      truncation=True,
      return_tensors="pt"
  )

  input_ids_tensor = encodings['input_ids']
  attention_mask_tensor = encodings['attention_mask']

  # preparation for alignment
  alignment_ids_list = []
  for i in range(len(characters)):
    alignment_ids = encodings.word_ids(batch_index=i)
    alignment_ids_list.append(alignment_ids)
  return input_ids_tensor, attention_mask_tensor, alignment_ids_list

```

In [81]:
def character_encode(characters, character_labels, label2id):
  encodings = tokenizer(
      characters,                    # List of List of characters
      is_split_into_words=True,
      padding='max_length',
      max_length=max_len,
      truncation=True,
      return_tensors="pt"
  )

  input_ids_tensor = encodings['input_ids']
  attention_mask_tensor = encodings['attention_mask']

  # preparation for alignment
  alignment_ids_list = []
  for i in range(len(characters)):
    alignment_ids = encodings.word_ids(batch_index=i)
    alignment_ids_list.append(alignment_ids)
  return input_ids_tensor, attention_mask_tensor, alignment_ids_list

- encoding the texts

```
train_input_ids_tensors, train_attention_mask_tensors, train_alignment_ids_list = character_encode(train_chars, train_labels, label2id)
dev_input_ids_tensors, dev_attention_mask_tensors, dev_alignment_ids_list = character_encode(dev_chars, dev_labels, label2id)
test_input_ids_tensors, test_attention_mask_tensors, test_alignment_ids_list = character_encode(test_chars, test_labels, label2id)
```

In [82]:
train_input_ids_tensors, train_attention_mask_tensors, train_alignment_ids_list = character_encode(train_chars, train_labels, label2id)
dev_input_ids_tensors, dev_attention_mask_tensors, dev_alignment_ids_list = character_encode(dev_chars, dev_labels, label2id)
test_input_ids_tensors, test_attention_mask_tensors, test_alignment_ids_list = character_encode(test_chars, test_labels, label2id)

#### aligning
It is very important to align tokens with labels after tokenising with BERT, since BERT performs tokenisation at the subword level.


- defining the function to perform the alignment

```
def align_labels(alignment_ids_list, character_labels):
  label_ids_list = []

  for alignment_ids, label_seq in zip(alignment_ids_list, character_labels):
      label_ids = []
      previous_word_idx = None

      for word_idx in alignment_ids:
          if word_idx is None:
              label_ids.append(-100)
          elif word_idx != previous_word_idx:
              label_ids.append(label2id[label_seq[word_idx]])
          else:
              label_ids.append(-100)
          previous_word_idx = word_idx

      label_ids_list.append(label_ids)

  label_tensor = torch.tensor(label_ids_list)
  return label_tensor
```

In [49]:
def align_labels(alignment_ids_list, character_labels):
  label_ids_list = []

  for alignment_ids, label_seq in zip(alignment_ids_list, character_labels):
      label_ids = []
      previous_word_idx = None

      for word_idx in alignment_ids:
          if word_idx is None:
              label_ids.append(-100)
          elif word_idx != previous_word_idx:
              label_ids.append(label2id[label_seq[word_idx]])
          else:
              label_ids.append(-100)
          previous_word_idx = word_idx

      label_ids_list.append(label_ids)

  label_tensor = torch.tensor(label_ids_list)
  return label_tensor

- performing the alignment

```
train_label_tensors=align_labels(train_alignment_ids_list, train_labels)
dev_label_tensors=align_labels(dev_alignment_ids_list, dev_labels)
test_label_tensors=align_labels(test_alignment_ids_list, test_labels)
```

In [83]:
train_label_tensors=align_labels(train_alignment_ids_list, train_labels)
dev_label_tensors=align_labels(dev_alignment_ids_list, dev_labels)
test_label_tensors=align_labels(test_alignment_ids_list, test_labels)

### creating `TensorDataset`

We combine the processed input tensors and label tensors into `TensorDataset` objects, one for each data split:

- **training set** → `train_dataset`
- **dev set** → `dev_dataset`
- **test set** → `test_dataset`

```
train_dataset = TensorDataset(train_input_ids_tensors, train_attention_mask_tensors, train_label_tensors)
dev_dataset = TensorDataset(dev_input_ids_tensors, dev_attention_mask_tensors, dev_label_tensors)
test_dataset = TensorDataset(test_input_ids_tensors, test_attention_mask_tensors, test_label_tensors)
```

In [84]:
train_dataset = TensorDataset(train_input_ids_tensors, train_attention_mask_tensors, train_label_tensors)
dev_dataset = TensorDataset(dev_input_ids_tensors, dev_attention_mask_tensors, dev_label_tensors)
test_dataset = TensorDataset(test_input_ids_tensors, test_attention_mask_tensors, test_label_tensors)

### converting Datasets into `DataLoader`

After creating `TensorDataset` objects for each data split, we wrap them with PyTorch `DataLoader` for efficient mini-batch loading during training and evaluation.

We specify the `batch_size` and whether to shuffle the data (shuffling is used only for training).

```
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
```

In [85]:
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## BERT




### Model defining

```
model = BertForTokenClassification.from_pretrained("bert-base-chinese",
                                                   num_labels=len(label2id))
torch.manual_seed(24)
```

In [86]:
model = BertForTokenClassification.from_pretrained("bert-base-chinese",
                                                   num_labels=len(label2id))
torch.manual_seed(24)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


- specifying the device

```
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
```

In [87]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


- setting the optimiser

```
optimizer = AdamW(model.parameters(), lr=5e-5)
```

In [88]:
optimizer = AdamW(model.parameters(), lr=5e-5)

### Training

```
train_losses = []
dev_losses = []
num_epochs = 1
model.to(device)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * input_ids.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)

    # validation
    model.eval()
    dev_loss = 0.0

    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            dev_loss += loss.item() * input_ids.size(0)

    avg_dev_loss = dev_loss / len(dev_loader.dataset)

    train_losses.append(avg_train_loss)
    dev_losses.append(avg_dev_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Dev Loss: {avg_dev_loss:.4f}")
```

In [89]:
train_losses = []
dev_losses = []
num_epochs = 1
model.to(device)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * input_ids.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)

    # validation
    model.eval()
    dev_loss = 0.0

    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            dev_loss += loss.item() * input_ids.size(0)

    avg_dev_loss = dev_loss / len(dev_loader.dataset)

    train_losses.append(avg_train_loss)
    dev_losses.append(avg_dev_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Dev Loss: {avg_dev_loss:.4f}")

Epoch 1/1 - Train Loss: 0.3073 - Dev Loss: 0.1611


### Predicting and evaluating


During prediction, the model outputs label IDs for all tokens, including special tokens like `[CLS]`, `[SEP]`, and `[PAD]`.  
These positions are marked as `-100` and should be removed before evaluation to correctly align predictions and labels.



- predicting

```
model.eval()

pred_labels_raw = []
true_labels_raw = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        # Note that the output here includes -100s, representing special tokens.
        for pred_seq, label_seq in zip(predictions, labels):
            pred_labels_raw.append(pred_seq)
            true_labels_raw.append(label_seq)
print(pred_labels_raw[0])
print(true_labels_raw[0])
```

In [90]:
model.eval()

pred_labels_raw = []
true_labels_raw = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        # Note that the output here includes -100s, representing special tokens.
        for pred_seq, label_seq in zip(predictions, labels):
            pred_labels_raw.append(pred_seq)
            true_labels_raw.append(label_seq)
print(pred_labels_raw[0])
print(true_labels_raw[0])

tensor([20, 20, 20, 20,  0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20,  0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20,  0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,  0, 10,  0, 10,
        10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20,  0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 10, 20, 20, 20, 20, 20, 20, 

- removing special IDs

```
pred_labels = []
true_labels = []

for pred_seq, label_seq in zip(pred_labels_raw, true_labels_raw):
    pred_seq = pred_seq.cpu().numpy().tolist()
    label_seq = label_seq.cpu().numpy().tolist()

    cleaned_preds = []
    for p, l in zip(pred_seq, label_seq):
      if l != -100:                         # If the true label is not a special token,
        cleaned_preds.append(p)             # save the predicted label in `cleaned_preds`.

    cleaned_labels = []
    for l in label_seq:
      if l != -100:
        cleaned_labels.append(l)

    pred_labels.append(cleaned_preds)
    true_labels.append(cleaned_labels)

print(true_labels[:2])
```

In [123]:
pred_labels_id = []
true_labels_id = []

for pred_seq, label_seq in zip(pred_labels_raw, true_labels_raw):
    pred_seq = pred_seq.cpu().numpy().tolist()
    label_seq = label_seq.cpu().numpy().tolist()

    cleaned_preds = []
    for p, l in zip(pred_seq, label_seq):
      if l != -100:                         # If the true label is not a special token,
        cleaned_preds.append(p)             # save the predicted label in `cleaned_preds`.

    cleaned_labels = []
    for l in label_seq:
      if l != -100:
        cleaned_labels.append(l)

    pred_labels_id.append(cleaned_preds)
    true_labels_id.append(cleaned_labels)

print(true_labels_id[:2])

[[20, 20, 20, 0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20], [20, 20, 20, 20, 2, 12, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 10, 10, 20, 20, 20, 20]]


- flattening the label list

This step is mainly required because `sklearn` evaluation functions expect flat lists of labels.

```
flattened_preds = []
flattened_trues = []

for pred_seq, label_seq in zip(pred_labels, true_labels):
  for p, l in zip(pred_seq, label_seq):
    flattened_preds.append(p)
    flattened_trues.append(l)
print(flattened_trues)
```

In [125]:
flattened_preds = []
flattened_trues = []

for pred_seq, label_seq in zip(pred_labels_id, true_labels_id):
  for p, l in zip(pred_seq, label_seq):
    flattened_preds.append(p)
    flattened_trues.append(l)
print(flattened_trues)

[20, 20, 20, 0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 2, 12, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 8, 18, 18, 20, 20, 20, 0, 10, 10, 20, 0, 10, 20, 20, 20, 20, 20, 0, 10, 20, 20, 20, 0, 10, 20, 20, 20, 20, 0, 10, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 0, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 2, 12, 12, 12, 20, 20, 20, 20

- evaluation

  In a sequence labelling task, we typically exclude the label `"O"` during evaluation. It is because correctly predicting `"O"` does not provide meaningful insight into the model's ability to identify named entities.

```
interesting_labels = []
for k in label2id:
  if k != "O":
    interesting_labels.append(k)

interesting_ids = []
for k in interesting_labels:
  interesting_ids.append(label2id[k])

evaluation=classification_report(
    flattened_trues,         # true labels
    flattened_preds,         # predicted labels
    labels=interesting_ids,
    target_names=interesting_labels,
    digits=4
)
print(evaluation)
```

In [126]:
interesting_labels = []
for k in label2id:
  if k != "O":
    interesting_labels.append(k)

interesting_ids = []
for k in interesting_labels:
  interesting_ids.append(label2id[k])

evaluation=classification_report(
    flattened_trues,         # true labels
    flattened_preds,         # predicted labels
    labels=interesting_ids,
    target_names=interesting_labels,
    digits=4
)
print(evaluation)

              precision    recall  f1-score   support

      B-BODY     0.8136    0.8840    0.8473       474
      B-CHEM     0.8455    0.7750    0.8087       120
      B-DISE     0.7785    0.8657    0.8198       134
      B-DRUG     1.0000    0.4000    0.5714        10
      B-EXAM     0.8409    0.7115    0.7708        52
      B-INST     0.0000    0.0000    0.0000        11
      B-SUPP     1.0000    0.5000    0.6667        14
      B-SYMP     0.7540    0.8060    0.7792       232
      B-TIME     1.0000    0.5000    0.6667         6
     B-TREAT     0.3448    0.3846    0.3636        26
      I-BODY     0.8483    0.8847    0.8661       607
      I-CHEM     0.8901    0.8679    0.8788       280
      I-DISE     0.8430    0.9034    0.8722       321
      I-DRUG     0.8333    0.4762    0.6061        21
      I-EXAM     0.8889    0.7123    0.7909       146
      I-INST     0.0000    0.0000    0.0000        25
      I-SUPP     0.9615    0.7812    0.8621        32
      I-SYMP     0.6834    

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [127]:
pred_labels_label=[]
for pred_label in pred_labels_id:
  label_list=[]
  for l in pred_label:
    label=id2label[l]
    label_list.append(label)
  pred_labels_label.append(label_list)

true_labels_label=[]
for true_label in true_labels_id:
  label_list=[]
  for l in true_label:
    label=id2label[l]
    label_list.append(label)
  true_labels_label.append(label_list)

- taking a look at the prediction result

```
print('Text:\n')
print(characters[0])
print('True labels:\n')
print(true_labels[0])
print('Predicted labels:\n')
print(pred_labels[0])
```

In [128]:
print('Text:\n')
print(test_chars[0])
print('True labels:\n')
print(true_labels_label[0])
print('Predicted labels:\n')
print(pred_labels_label[0])

Text:

['您', '好', '，', '紅', '血', '球', '分', '佈', '寬', '度', '有', '兩', '種', '參', '考', '方', '式', '，', '分', '別', '是', '紅', '血', '球', '分', '佈', '寬', '度', '變', '異', '數', '（', 'Ｃ', 'Ｖ', '）', '與', '紅', '血', '球', '分', '佈', '寬', '度', '標', '準', '差', '（', 'Ｓ', 'Ｄ', '）', '。']
True labels:

['O', 'O', 'O', 'B-BODY', 'I-BODY', 'I-BODY', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-BODY', 'I-BODY', 'I-BODY', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-BODY', 'I-BODY', 'I-BODY', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Predicted labels:

['O', 'O', 'O', 'B-BODY', 'I-BODY', 'I-BODY', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-BODY', 'I-BODY', 'I-BODY', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-BODY', 'I-BODY', 'I-BODY', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


## The fine-tuned model

- saving the fine-tuned `BERT`

```
model.save_pretrained("my_finetuned_bert/")
tokenizer.save_pretrained("my_finetuned_bert/")
```

In [62]:
model.save_pretrained("my_finetuned_bert/")
tokenizer.save_pretrained("my_finetuned_bert/")

('my_finetuned_bert/tokenizer_config.json',
 'my_finetuned_bert/special_tokens_map.json',
 'my_finetuned_bert/vocab.txt',
 'my_finetuned_bert/added_tokens.json',
 'my_finetuned_bert/tokenizer.json')

- importing the fine-tuned `BERT`

```
fine_tuned_model = BertForTokenClassification.from_pretrained("my_finetuned_bert/")
tokenizer = BertTokenizerFast.from_pretrained("my_finetuned_bert/")
fine_tuned_model.eval()
```

In [95]:
fine_tuned_model = BertForTokenClassification.from_pretrained("my_finetuned_bert/")
tokenizer = BertTokenizerFast.from_pretrained("my_finetuned_bert/")
fine_tuned_model.eval()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

- defining the text

```
input_text = "三餐正常吃，卻還是嘴饞？用餐後特別想睡？或許是「醣類疲勞」在搞鬼。日本糖尿病專科醫師山田悟提醒，用餐後高血糖及血糖震盪可能引起醣類疲勞，且幾乎人人都有醣類疲勞的問題，可能會出現哪些症狀？"
new_text = list(input_text)  # character-level
```

In [96]:
input_text = "三餐正常吃，卻還是嘴饞？用餐後特別想睡？或許是「醣類疲勞」在搞鬼。日本糖尿病專科醫師山田悟提醒，用餐後高血糖及血糖震盪可能引起醣類疲勞，且幾乎人人都有醣類疲勞的問題，可能會出現哪些症狀？"
new_text = list(input_text)  # character-level

- tokenising

```
encoding = tokenizer(
    new_text,
    is_split_into_words=True,
    return_tensors="pt",
    padding=True,
    truncation=True
)
encoding.keys()
```

In [97]:
encoding = tokenizer(
    new_text,
    is_split_into_words=True,
    return_tensors="pt",
    padding=True,
    truncation=True
)
encoding.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

- predicting

```
fine_tuned_model.to(device)
fine_tuned_model.eval()
with torch.no_grad():
    outputs = fine_tuned_model(encoding['input_ids'].to(device),
                               encoding['attention_mask'].to(device))
    predictions = outputs.logits.argmax(dim=-1).squeeze().cpu().tolist()
print(len(predictions))
print(predictions[:11])
```

In [103]:
fine_tuned_model.to(device)
fine_tuned_model.eval()
with torch.no_grad():
    outputs = fine_tuned_model(encoding['input_ids'].to(device),
                               encoding['attention_mask'].to(device))
    predictions = outputs.logits.argmax(dim=-1).squeeze().cpu().tolist()
print(len(predictions))
print(predictions[:11])

95
[20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 7]


- removing special tokens and converting IDs into labels

```
word_ids = encoding.word_ids(batch_index=0)

predicted_labels = []
previous_word_idx = None

for idx, word_idx in enumerate(word_ids):
    if word_idx is None:
        continue  # skip special tokens like CLS/SEP/PAD
    if word_idx != previous_word_idx:
        predicted_labels.append(id2label[predictions[idx]])
    previous_word_idx = word_idx   # updating `previous_word_idx`
print(len(predicted_labels))
print(predicted_labels[:10])
```

In [99]:
word_ids = encoding.word_ids(batch_index=0)

predicted_labels = []
previous_word_idx = None

for idx, word_idx in enumerate(word_ids):
    if word_idx is None:
        continue  # skip special tokens like CLS/SEP/PAD
    if word_idx != previous_word_idx:
        predicted_labels.append(id2label[predictions[idx]])
    previous_word_idx = word_idx   # updating `previous_word_idx`
print(len(predicted_labels))
print(predicted_labels[:10])

93
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SYMP']


- printing out the results

```
print("字\t預測標籤")
print("─" * 30)
for char, label in zip(new_text, predicted_labels):
    print(f"{char}\t{label}")
```

In [100]:
print("字\t預測標籤")
print("─" * 30)
for char, label in zip(new_text, predicted_labels):
    print(f"{char}\t{label}")

字	預測標籤
──────────────────────────────
三	O
餐	O
正	O
常	O
吃	O
，	O
卻	O
還	O
是	O
嘴	B-SYMP
饞	I-SYMP
？	O
用	O
餐	O
後	O
特	O
別	O
想	O
睡	O
？	O
或	O
許	O
是	O
「	O
醣	B-BODY
類	I-CHEM
疲	I-SYMP
勞	I-SYMP
」	O
在	O
搞	O
鬼	O
。	O
日	O
本	O
糖	B-DISE
尿	I-DISE
病	I-DISE
專	O
科	O
醫	O
師	O
山	O
田	O
悟	O
提	O
醒	O
，	O
用	O
餐	O
後	O
高	B-DISE
血	I-DISE
糖	I-CHEM
及	O
血	B-BODY
糖	I-CHEM
震	B-SYMP
盪	I-SYMP
可	O
能	O
引	O
起	O
醣	B-BODY
類	I-CHEM
疲	B-SYMP
勞	I-SYMP
，	O
且	O
幾	O
乎	O
人	O
人	O
都	O
有	O
醣	B-BODY
類	I-CHEM
疲	B-SYMP
勞	I-SYMP
的	O
問	O
題	O
，	O
可	O
能	O
會	O
出	O
現	O
哪	O
些	O
症	O
狀	O
？	O
